# 🚀 Hướng Dẫn Thực Nghiệm Phân Phối Trên Google Colab

> **Lưu ý quan trọng trước khi bắt đầu:**
> 1. **Runtime GPU:** Chọn **T4 GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).
> 2. **Lối tắt Drive:** Hãy tạo lối tắt (Shortcut) thư mục `KLTN-2026-testingNtraining` vào `My Drive` của bạn trước khi chạy.

## Bước 1: Kết nối Google Drive & Kiểm tra Thư mục Dùng Chung

**Hướng dẫn tạo lối tắt (Shortcut):**
1. Mở Google Drive trên trình duyệt web -> Vào mục **"Được chia sẻ với tôi" (Shared with me)**.
2. Click chuột phải vào thư mục **`KLTN-2026-testingNtraining`** -> Chọn **"Thêm lối tắt vào Drive" (Add shortcut to Drive)**.
3. Chọn lưu vào **"My Drive" (Drive của tôi)** -> Bấm **Add (Thêm)**.
4. Chạy cell dưới đây để kiểm tra kết nối.

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

SHARED_DRIVE = "/content/drive/MyDrive/KLTN-2026-testingNtraining"

if not os.path.exists(SHARED_DRIVE):
    print("❌ CHƯA TÌM THẤY THƯ MỤC CHUNG!")
    print("👉 Hãy vào Google Drive web -> 'Được chia sẻ với tôi' -> Chuột phải vào 'KLTN-2026-testingNtraining' -> 'Thêm lối tắt vào Drive' -> chọn 'My Drive' rồi chạy lại cell này.")
else:
    print(f"✅ Đã kết nối thành công với thư mục chung: {SHARED_DRIVE}")
    # Tạo sẵn các thư mục con trong runs nếu chưa có
    os.makedirs(f"{SHARED_DRIVE}/runs/baseline", exist_ok=True)
    os.makedirs(f"{SHARED_DRIVE}/runs/cbam_backbone", exist_ok=True)
    os.makedirs(f"{SHARED_DRIVE}/runs/evaluation", exist_ok=True)
    print("✅ Các thư mục runs/baseline, runs/cbam_backbone, runs/evaluation đã sẵn sàng!")

## Bước 2: Clone Source Code từ GitHub & Cài đặt Thư viện

In [ ]:
# 1. Clone repository chính thức
!git clone https://github.com/mizzhau/yolo11n-cbam-mvtec-defect-detection.git
%cd yolo11n-cbam-mvtec-defect-detection

# 2. Cài đặt các thư viện cần thiết
!pip install -q ultralytics albumentations tabulate

## Bước 3: Ghép Part Nhị Phân & Giải Nén Dữ Liệu Lên SSD Colab
- Dữ liệu gồm 5 file part (`.bin`) được nối lại thành file nén tạm thời trên SSD Colab.
- Sử dụng công cụ `7z` tự động giải nén đa năng (nhận diện chính xác cả định dạng Zip và 7z).
- Giải nén vào ổ SSD `/content/` giúp quá trình huấn luyện nạp ảnh đạt tốc độ tối đa, không bị nghẽn mạng.

In [ ]:
# 1. Ghép 5 part nhị phân từ Drive chung thành archive tạm trên SSD Colab
print("⏳ Đang ghép 5 part nhị phân...")
!cat /content/drive/MyDrive/KLTN-2026-testingNtraining/dataset/mvtec_augmented.part*.bin > /content/archive_temp

# 2. Giải nén vào đúng thư mục dữ liệu quy ước của repo
print("⏳ Đang giải nén dữ liệu vào data/processed/split_70_15_15_augmented/...")
!mkdir -p data/processed/split_70_15_15_augmented
!7z x /content/archive_temp -odata/processed/split_70_15_15_augmented -y > /dev/null

# 3. Xóa file archive tạm để tiết kiệm dung lượng SSD
!rm -f /content/archive_temp

print("✅ Hoàn tất giải nén! Kiểm tra thư mục dữ liệu:")
!ls -la data/processed/split_70_15_15_augmented

## Bước 4: Thực Nghiệm Huấn Luyện (20 Epochs, lưu Checkpoint từng Epoch)

> **Phân công thành viên:**
> - **Thành viên A:** Chạy **Cell 4A (Baseline YOLO11n)**.
> - **Thành viên B:** Chạy **Cell 4B (YOLO11n + CBAM Backbone)**.
> - Cả 2 có thể mở 2 notebook độc lập trên 2 tài khoản Colab để chạy song song.
> - Checkpoints từng epoch (`epoch1.pt`, `epoch2.pt`, ..., `best.pt`) và metrics `results.csv` sẽ được tự động lưu trực tiếp vào Drive chung.

### Cell 4A: Huấn Luyện BASELINE (Dành cho Thành viên chạy YOLO11n)

In [ ]:
# Chạy huấn luyện Baseline 20 epochs, tự động lưu checkpoint mỗi epoch sang Drive chung
!python src/training/train_baseline.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 20 \
    --batch 16 \
    --imgsz 640 \
    --save_period 1 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline \
    --name train

### Cell 4B: Huấn Luyện MÔ HÌNH CHÍNH (Dành cho Thành viên chạy YOLO11n + CBAM)

In [ ]:
# Chạy huấn luyện CBAM Backbone 20 epochs, tự động lưu checkpoint mỗi epoch sang Drive chung
!python src/training/train_cbam.py \
    --data configs/data/mvtec_70_15_15_augmented.yaml \
    --epochs 20 \
    --batch 16 \
    --imgsz 640 \
    --save_period 1 \
    --project /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/cbam_backbone \
    --name train

## Bước 5: Đánh Giá So Sánh & Xuất Biểu Đồ Tự Động Về Drive Chung
*Sau khi cả 2 mô hình hoàn thành 20 epochs, bất kỳ thành viên nào cũng có thể chạy cell này để tổng hợp kết quả.*

In [ ]:
# Xuất biểu đồ Learning Curves, Bar Chart và bảng Markdown so sánh vào thư mục chung
!python src/evaluation/plot_comparison.py \
    --baseline_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/baseline/train \
    --cbam_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/cbam_backbone/train \
    --output_dir /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/evaluation

print("\n🎉 Hoàn tất đánh giá so sánh! Biểu đồ và báo cáo đã được lưu tại:")
!ls -la /content/drive/MyDrive/KLTN-2026-testingNtraining/runs/evaluation